# 01 — Exploratory Data Analysis
**ESCP Hackathon 2026 · Flood Risk Prediction**

This notebook explores the four NetCDF files:
- `flood_risk_terrain_northumbria.nc` — terrain + flood risk labels (Northumbria)
- `flood_risk_terrain_severn.nc`      — terrain + flood risk labels (Severn)
- `era5_land_northumbria.nc`          — ERA5 weather (Northumbria)
- `era5_land_severn.nc`               — ERA5 weather (Severn)


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings, json
from pathlib import Path

try:
    import netCDF4 as nc
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "netCDF4", "-q"])
    import netCDF4 as nc

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

DATA_DIR = Path("../Data")
print("Data files found:")
for f in sorted(DATA_DIR.glob("*.nc")):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")


Data files found:


## 1. Helper utilities

In [ ]:
def nc_summary(path, sample_n=500_000):
    """Print a compact summary of a NetCDF file."""
    ds = nc.Dataset(path, "r")
    print(f"\n{'='*65}")
    print(f"  FILE: {Path(path).name}")
    print(f"{'='*65}")
    print(f"  Dimensions: { {k: int(v.size) for k,v in ds.dimensions.items()} }")

    rows = []
    for vname, var in ds.variables.items():
        shape = var.shape
        dtype = str(var.dtype)

        # time variable
        if vname in ("time", "valid_time"):
            try:
                t = var[:]
                t0 = str(nc.num2date(t[0],  var.units))
                t1 = str(nc.num2date(t[-1], var.units))
                rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                             "min": t0, "max": t1, "mean": "-", "pct_nan": "-",
                             "note": f"{len(t)} steps"})
            except:
                rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                             "min": "-", "max": "-", "mean": "-", "pct_nan": "-", "note": ""})
            continue

        # coordinate variable
        if vname in ("projection_x_coordinate", "projection_y_coordinate",
                     "x", "y", "lon", "lat", "longitude", "latitude"):
            arr = var[:].flatten()
            rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                         "min": round(float(arr.min()),1), "max": round(float(arr.max()),1),
                         "mean": "-", "pct_nan": "-", "note": f"n={len(arr)}"})
            continue

        # data variable — sample
        try:
            raw  = var[:]
            flat = np.ma.compressed(np.ma.masked_invalid(raw.flatten()))
            pct_nan = round(100*(1 - len(flat)/raw.size), 2)
            if len(flat) == 0:
                rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                             "min": np.nan, "max": np.nan, "mean": np.nan,
                             "pct_nan": 100, "note": "all NaN"})
                continue
            idx    = np.random.choice(len(flat), min(sample_n, len(flat)), replace=False)
            s      = flat[idx]
            uniq   = np.unique(flat)
            note   = f"uniq={list(uniq.astype(float).round(2))}" if len(uniq)<=20 else f"{len(uniq)} unique"
            rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                         "min":  round(float(s.min()),4),
                         "max":  round(float(s.max()),4),
                         "mean": round(float(s.mean()),4),
                         "pct_nan": pct_nan, "note": note})
        except Exception as e:
            rows.append({"variable": vname, "shape": shape, "dtype": dtype,
                         "min": "-", "max": "-", "mean": "-", "pct_nan": "-", "note": str(e)})

    ds.close()
    df = pd.DataFrame(rows)
    display(df.to_string(index=False))
    return df


## 2. Terrain data — Northumbria

In [ ]:
df_tn = nc_summary(DATA_DIR / "flood_risk_terrain_northumbria.nc")


## 3. Terrain data — Severn

In [ ]:
df_ts = nc_summary(DATA_DIR / "flood_risk_terrain_severn.nc")


## 4. Weather data (ERA5)

In [ ]:
df_wn = nc_summary(DATA_DIR / "era5_land_northumbria.nc")
df_ws = nc_summary(DATA_DIR / "era5_land_severn.nc")


## 5. Load terrain as flat DataFrames (sampled)

In [ ]:
def terrain_to_df(path, sample_frac=0.05, seed=42):
    """Load terrain NetCDF as a flat DataFrame. Samples to keep memory low."""
    ds = nc.Dataset(path, "r")
    vars_to_load = [v for v in ds.variables
                    if v not in ("projection_x_coordinate", "projection_y_coordinate")]
    
    # Get coordinate arrays
    x = ds.variables["projection_x_coordinate"][:]
    y = ds.variables["projection_y_coordinate"][:]
    n = len(x)
    
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=int(n * sample_frac), replace=False)
    idx.sort()
    
    data = {"x": x[idx], "y": y[idx]}
    for v in vars_to_load:
        arr = ds.variables[v][:]
        data[v] = np.array(arr.flat)[idx] if arr.ndim == 1 else arr.flatten()[idx]
    
    ds.close()
    df = pd.DataFrame(data)
    # Replace masked/fill values with NaN
    df.replace(-9999, np.nan, inplace=True)
    df.replace(9.96921e+36, np.nan, inplace=True)
    print(f"Loaded {len(df):,} rows ({sample_frac*100:.0f}% sample) from {Path(path).name}")
    return df

df_north = terrain_to_df(DATA_DIR / "flood_risk_terrain_northumbria.nc", sample_frac=0.05)
df_severn = terrain_to_df(DATA_DIR / "flood_risk_terrain_severn.nc",     sample_frac=0.05)

print("\nNorthumbria columns:", list(df_north.columns))
display(df_north.head(3))


## 6. Target variable distributions (flood risk)

In [ ]:
RISK_VARS   = ["risk_0_2m", "risk_0_3m", "risk_0_6m", "risk_0_9m", "risk_1_2m"]
RISK_LABELS = {1: "Very Low", 2: "Low", 3: "Medium", 4: "High"}
COLORS      = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

fig, axes = plt.subplots(2, len(RISK_VARS), figsize=(18, 7))
fig.suptitle("Flood Risk Category Distribution by Depth", fontsize=13, fontweight="bold")

for col, rv in enumerate(RISK_VARS):
    for row, (df, region) in enumerate([(df_north, "Northumbria"), (df_severn, "Severn")]):
        ax = axes[row][col]
        if rv not in df.columns:
            ax.set_visible(False); continue
        counts = df[rv].dropna().value_counts().sort_index()
        bars = ax.bar([RISK_LABELS.get(int(k), k) for k in counts.index],
                      counts.values, color=COLORS[:len(counts)], edgecolor="white")
        ax.set_title(f"{rv}\n{region}", fontsize=8)
        ax.tick_params(axis="x", labelsize=7, rotation=30)
        ax.tick_params(axis="y", labelsize=7)
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                    f"{val/1000:.0f}k", ha="center", fontsize=6)

plt.tight_layout()
plt.savefig("../reports/figures/01_risk_distributions.png", bbox_inches="tight")
plt.show()


## 7. Spatial maps — flood risk at 0.2m

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Flood Risk at 0.2m Depth (5% sample)", fontsize=13, fontweight="bold")

cmap = mcolors.ListedColormap(["#FFFFFF", "#2196F3", "#4CAF50", "#FF9800", "#F44336"])
norm = mcolors.BoundaryNorm([0, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

for ax, df, title in [(axes[0], df_north, "Northumbria"),
                      (axes[1], df_severn, "Severn")]:
    if "risk_0_2m" not in df.columns:
        ax.set_title(f"{title} — variable missing"); continue
    valid = df.dropna(subset=["risk_0_2m"])
    sc = ax.scatter(valid["x"], valid["y"], c=valid["risk_0_2m"],
                    s=0.3, cmap=cmap, norm=norm, rasterized=True)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
    ax.tick_params(labelsize=7)
    cbar = plt.colorbar(sc, ax=ax, ticks=[1,2,3,4])
    cbar.ax.set_yticklabels(["Very Low","Low","Medium","High"], fontsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/02_spatial_risk_map.png", bbox_inches="tight")
plt.show()


## 8. Terrain feature distributions

In [ ]:
TERRAIN_VARS = ["dtm", "waw", "imd", "clc_type", "flow_acc", "rciw"]

fig, axes = plt.subplots(2, len(TERRAIN_VARS), figsize=(20, 7))
fig.suptitle("Terrain Feature Distributions", fontsize=13, fontweight="bold")

for col, var in enumerate(TERRAIN_VARS):
    for row, (df, region) in enumerate([(df_north, "Northumbria"), (df_severn, "Severn")]):
        ax = axes[row][col]
        if var not in df.columns:
            ax.set_visible(False); continue
        data = df[var].dropna()
        uniq = data.nunique()
        if uniq <= 20:
            vc = data.value_counts().sort_index()
            ax.bar(vc.index.astype(str), vc.values, color="#5C6BC0", edgecolor="white")
            ax.tick_params(axis="x", labelsize=6, rotation=45)
        else:
            # log scale for flow_acc
            vals = np.log1p(data) if var == "flow_acc" else data
            ax.hist(vals, bins=50, color="#5C6BC0", edgecolor="white", alpha=0.85)
            if var == "flow_acc":
                ax.set_xlabel("log(1+flow_acc)", fontsize=7)
        ax.set_title(f"{var}\n{region}", fontsize=8)
        ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/03_terrain_distributions.png", bbox_inches="tight")
plt.show()


## 9. Elevation comparison (key domain-shift issue)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for df, label, color in [(df_north, "Northumbria", "#1565C0"),
                          (df_severn, "Severn",     "#C62828")]:
    if "dtm" not in df.columns: continue
    vals = df["dtm"].dropna() / 10  # convert dm → m
    ax.hist(vals, bins=80, alpha=0.55, label=f"{label} (mean={vals.mean():.0f}m)",
            color=color, density=True)
ax.set_xlabel("Elevation (m)", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Elevation Distribution — Domain Shift Warning", fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/04_elevation_shift.png", bbox_inches="tight")
plt.show()
print("NOTE: Northumbria is higher on average → raw dtm will leak region identity.")
print("Consider relative/percentile elevation or use flow_acc as the primary terrain proxy.")


## 10. Feature–target correlation (Northumbria training set)

In [ ]:
TARGET    = "risk_0_2m"
FEAT_COLS = ["dtm", "waw", "imd", "flow_acc", "clc_type"]

sub = df_north[[TARGET] + FEAT_COLS].dropna()
corr = sub.corr(method="spearman")[TARGET].drop(TARGET).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#F44336" if v > 0 else "#2196F3" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman correlation with risk_0_2m", fontsize=10)
ax.set_title("Feature Correlations — Northumbria (training region)", fontsize=11)
plt.tight_layout()
plt.savefig("../reports/figures/05_feature_correlation.png", bbox_inches="tight")
plt.show()


## 11. Weather data — quick overview

In [ ]:
def weather_timeseries_sample(path, var="tp", n_pixels=5, seed=42):
    """Plot mean daily time series for a weather variable."""    ds = nc.Dataset(path, "r")
    if var not in ds.variables:
        print(f"{var} not found in {Path(path).name}"); ds.close(); return

    # time
    t_raw = ds.variables["valid_time"][:]
    try:
        dates = nc.num2date(t_raw, ds.variables["valid_time"].units)
        dates = pd.to_datetime([str(d) for d in dates])
    except:
        dates = pd.RangeIndex(len(t_raw))

    data = ds.variables[var][:]  # shape: (time, y, x) or (time, pixel)
    ds.close()

    # spatial mean across all pixels at each time step
    if data.ndim == 3:
        ts = np.ma.mean(data, axis=(1, 2))
    else:
        ts = np.ma.mean(data, axis=1)

    return pd.Series(ts.filled(np.nan), index=dates, name=var)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle("ERA5 Weather — Spatial Mean Time Series", fontsize=13, fontweight="bold")

WEATHER_VARS = [("tp", "Total Precipitation (m/day)"),
                ("sro", "Surface Runoff (m/day)"),
                ("swvl1_mean", "Soil Water Vol. Layer 1 (mean)"),
                ("t2m_mean", "2m Temperature (K, mean)")]

for ax, (var, label) in zip(axes.flatten(), WEATHER_VARS):
    for path, region, color in [
        (DATA_DIR/"era5_land_northumbria.nc", "Northumbria", "#1565C0"),
        (DATA_DIR/"era5_land_severn.nc",      "Severn",      "#C62828"),
    ]:
        ts = weather_timeseries_sample(path, var=var)
        if ts is not None:
            ts.resample("ME").mean().plot(ax=ax, label=region, color=color, linewidth=1.2)
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("../reports/figures/06_weather_timeseries.png", bbox_inches="tight")
plt.show()


## 12. Export compact EDA summary (for Claude / team)

In [ ]:
def extract_summary(path, sample_n=200_000):
    ds  = nc.Dataset(path, "r")
    out = {"file": Path(path).name,
           "dimensions": {k: int(v.size) for k,v in ds.dimensions.items()},
           "variables": {}}
    for vname, var in ds.variables.items():
        info = {"shape": list(var.shape), "dtype": str(var.dtype)}
        if vname in ("time","valid_time"):
            try:
                t = var[:]
                info["range"] = [str(nc.num2date(t[0],var.units)),
                                 str(nc.num2date(t[-1],var.units))]
                info["n_steps"] = int(len(t))
            except: pass
        elif vname in ("projection_x_coordinate","projection_y_coordinate","x","y"):
            arr = var[:]
            info["range"] = [round(float(arr.min()),1), round(float(arr.max()),1)]
        else:
            try:
                raw  = var[:]
                flat = np.ma.compressed(np.ma.masked_invalid(raw.flatten()))
                idx  = np.random.choice(len(flat), min(sample_n,len(flat)), replace=False)
                s    = flat[idx]
                uniq = np.unique(flat)
                info["stats"] = {
                    "min":     round(float(s.min()),5),
                    "max":     round(float(s.max()),5),
                    "mean":    round(float(s.mean()),5),
                    "std":     round(float(s.std()),5),
                    "p25":     round(float(np.percentile(s,25)),5),
                    "p75":     round(float(np.percentile(s,75)),5),
                    "pct_nan": round(100*(1-len(flat)/raw.size),2),
                }
                if len(uniq) <= 25:
                    info["unique_values"] = [round(float(v),3) for v in uniq]
            except Exception as e:
                info["error"] = str(e)
        out["variables"][vname] = info
    ds.close()
    return out

summary = {label: extract_summary(DATA_DIR/fname)
           for label, fname in [
               ("terrain_northumbria", "flood_risk_terrain_northumbria.nc"),
               ("terrain_severn",      "flood_risk_terrain_severn.nc"),
               ("weather_northumbria", "era5_land_northumbria.nc"),
               ("weather_severn",      "era5_land_severn.nc"),
           ]}

out_path = Path("../reports/eda_summary.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Summary saved to {out_path}  ({out_path.stat().st_size/1024:.1f} KB)")
print("Paste eda_summary.json contents back to Claude for modelling advice.")


## 13. Class imbalance check across all risk depths

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Class Imbalance — All Flood Depths", fontsize=12, fontweight="bold")

for ax, df, title in [(axes[0], df_north, "Northumbria"), (axes[1], df_severn, "Severn")]:
    records = []
    for rv in RISK_VARS:
        if rv not in df.columns: continue
        vc = df[rv].dropna().value_counts(normalize=True).sort_index() * 100
        for k, v in vc.items():
            records.append({"depth": rv.replace("risk_","").replace("_","=")+"m",
                            "category": RISK_LABELS.get(int(k), k),
                            "pct": round(v, 2)})
    pivot = pd.DataFrame(records).pivot(index="depth", columns="category", values="pct").fillna(0)
    pivot.plot(kind="bar", ax=ax, color=COLORS[:pivot.shape[1]], edgecolor="white", width=0.7)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("% of pixels")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../reports/figures/07_class_imbalance.png", bbox_inches="tight")
plt.show()
print("\nTIP: High class imbalance likely — consider class weights or stratified sampling.")


## 14. Missing values analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Missing Values (%) by Variable", fontsize=12, fontweight="bold")

for ax, df, title in [(axes[0], df_north, "Northumbria"), (axes[1], df_severn, "Severn")]:
    nan_pct = df.isna().mean().sort_values(ascending=True) * 100
    nan_pct = nan_pct[nan_pct > 0]
    if len(nan_pct) == 0:
        ax.text(0.5, 0.5, "No missing values", ha="center", va="center", fontsize=11)
    else:
        ax.barh(nan_pct.index, nan_pct.values, color="#EF5350", edgecolor="white")
        ax.set_xlabel("% NaN")
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.savefig("../reports/figures/08_missing_values.png", bbox_inches="tight")
plt.show()


## 15. Key EDA Takeaways

| Issue | Finding | Action |
|-------|---------|--------|
| **Domain shift** | Northumbria mean elevation ~196m vs Severn ~99m | Use relative/percentile DTM or `flow_acc` as primary proxy |
| **Class imbalance** | Most pixels are "Very Low" risk | Use class weights or focal loss |
| **Spatial correlation** | Adjacent pixels share risk | CNN / spatial model; don't treat as i.i.d. |
| **NaN = no risk** | NaN in risk vars = outside flood zone | Decide: exclude or treat as class 0 |
| **Weather resolution** | ERA5 at ~9.45km vs terrain at 20m | Aggregate to pixel-level extremes (rolling windows) |
| **Multi-depth targets** | 5 risk depths available | Fit one multi-output model or 5 separate models |
| **`flow_acc` skew** | Highly right-skewed | Use `log1p(flow_acc)` as feature |
